# Middleware 深度解析

本笔记从概念到源码，系统梳理 LangChain 中间件（Middleware）的设计与实现。

## 1. 中间件的本质：管道拦截器

中间件（Middleware）是一种**在请求和响应之间插入处理逻辑**的设计模式，核心思想是：

```
请求 → [中间件1] → [中间件2] → [中间件N] → 核心处理 → 响应
        ↑               ↑                              |
        └───────────────┴──────────────────────────────┘
                    响应也可以反向经过
```

### 三种经典形态

**1. 洋葱模型（双向拦截）**  
以 Express.js / Koa 为代表：
```
请求进入 → MW1 前置 → MW2 前置 → 核心
                                   ↓
响应返回 ← MW1 后置 ← MW2 后置 ←─┘
```
每个中间件可以在**调用前后**各做一次处理（如计时、日志、错误捕获）。

**2. 单向过滤器（管道模式）**  
以 Linux pipe / Java Servlet Filter 为代表：
```
输入 | filter1 | filter2 | filter3 | 输出
```
数据只往一个方向流动，每一步做转换或过滤。

**3. 钩子模型（Hook）**  
以 LangChain / Git hooks 为代表：
```
事件触发 → before_hook → 核心执行 → after_hook
```
在特定生命周期节点注入逻辑，核心流程不感知钩子存在。

### 统一抽象：横切关注点分离

所有中间件都解决同一个问题：**把横切关注点（Cross-cutting Concerns）从业务逻辑中剥离出来**。

| 横切关注点 | 典型中间件 |
|-----------|----------|
| 认证/鉴权 | JWT 验证、API Key 检查 |
| 日志/追踪 | 请求日志、链路追踪 |
| 数据转换 | 压缩、加密、序列化 |
| 流量控制 | 限流、熔断、重试 |
| 上下文管理 | LangChain 的消息压缩/裁剪 |

**设计要点：**
- **可组合**：多个中间件可自由叠加，顺序即逻辑
- **对核心透明**：业务核心不需要知道中间件的存在
- **单一职责**：每个中间件只做一件事
- **可插拔**：加减中间件不影响核心代码

## 2. LangChain 中间件机制

LangChain 的 Agent 中间件通过 `create_agent` 的 `middleware=[]` 参数注入，支持两种形式：

### 2.1 内置中间件类（`AgentMiddleware` 子类）

继承 `AgentMiddleware`，覆盖对应的 hook 方法：

| Hook | 触发时机 | 循环 |
|------|---------|------|
| `before_agent` | Agent 启动时，**仅一次** | 否 |
| `before_model` | 每次调用 LLM 前 | 是 |
| `wrap_model_call` | 包裹 LLM 调用（洋葱） | 是 |
| `after_model` | 每次调用 LLM 后 | 是 |
| `wrap_tool_call` | 包裹工具调用（洋葱） | 是 |
| `after_agent` | Agent 结束时，**仅一次** | 否 |

### 2.2 函数式中间件（`@before_agent` 装饰器）

用装饰器将普通函数提升为中间件：

```python
from langchain.agents.middleware import before_agent

@before_agent
def my_middleware(state: AgentState, runtime: Runtime) -> dict | None:
    # 返回 state 的修改 dict，或 None 表示不修改
    return {"messages": [RemoveMessage(id=m.id) for m in unwanted]}
```

### 2.3 整体执行流程

```
invoke()
   ↓
[before_agent]   ← 仅一次，适合初始化/消息压缩
   ↓
┌──[before_model] ← 每轮循环，适合消息预处理
│     ↓
│  [model]        ← LLM 调用（wrap_model_call 在此包裹）
│     ↓
│  [after_model]  ← 每轮循环，适合结果后处理
│     ↓
│  [tools]        ← 执行工具调用（wrap_tool_call 在此包裹）
└──────────────── ← 有 tool_call 则循环，否则退出
   ↓
[after_agent]    ← 仅一次，适合清理/总结
```

## 3. 用法示例

### 3.1 SummarizationMiddleware：自动压缩历史消息

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-4o-mini",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 100),  # 超过 100 tokens 时触发压缩
            keep=("messages", 1),     # 保留最近 1 条原始消息
        )
    ],
)

**SummarizationMiddleware 工作原理：**

- 挂载在 `before_agent` hook，整个会话**只触发一次**
- 当 token 数超过阈值，将旧消息压缩为一条结构化 `HumanMessage`，格式如下：

```
## SESSION INTENT   ← 用户意图
## SUMMARY          ← 对话要点
## ARTIFACTS        ← 产出物
## NEXT STEPS       ← 后续步骤
```

- 压缩消息附带 `additional_kwargs={'lc_source': 'summarization'}` 标记
- **checkpointer 完整历史不变**，只有转发给 LLM 的上下文被压缩

### 3.2 函数式中间件：裁剪 ToolMessage

In [ ]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage, ToolMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent

@before_agent
def trim_tool_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """在 Agent 启动前移除所有 ToolMessage，避免历史工具输出干扰 LLM"""
    tool_messages = [m for m in state["messages"] if isinstance(m, ToolMessage)]
    if not tool_messages:
        return None  # 无需修改
    # RemoveMessage 是 LangGraph 的特殊原语，通过 id 从 state 中删除消息
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}


agent = create_agent(
    model="gpt-4o-mini",
    checkpointer=InMemorySaver(),
    middleware=[trim_tool_messages],
)

## 4. 底层实现：中间件 = LangGraph 节点

`create_agent` 在底层构建的是一个 **LangGraph `StateGraph`**，中间件的每个 hook 被编译成图中的独立**节点（Node）**，通过边（Edge）连接成执行链。

源码位置：`langchain/agents/factory.py`

---

### Step 1：Hook 分类（factory.py:913）

```python
# 用 is not 比较类方法，判断子类是否覆盖了基类的空实现
# 没覆盖 = 不注册节点，零成本
middleware_w_before_agent = [
    m for m in middleware
    if m.__class__.before_agent is not AgentMiddleware.before_agent
    or m.__class__.abefore_agent is not AgentMiddleware.abefore_agent
]
middleware_w_before_model = [...]
middleware_w_after_model  = [...]
middleware_w_after_agent  = [...]
middleware_w_wrap_model_call = [...]  # 洋葱包裹，不作为独立节点
middleware_w_wrap_tool_call  = [...]  # 洋葱包裹，不作为独立节点
```

---

### Step 2：注册为 Graph 节点（factory.py:1322）

```python
for m in middleware:
    if m.__class__.before_agent is not AgentMiddleware.before_agent:
        # 包装为 RunnableCallable（同时支持 sync/async）
        graph.add_node(
            f"{m.name}.before_agent",
            RunnableCallable(m.before_agent, m.abefore_agent)
        )
    if m.__class__.before_model ...:
        graph.add_node(f"{m.name}.before_model", ...)
    if m.__class__.after_model ...:
        graph.add_node(f"{m.name}.after_model", ...)
    if m.__class__.after_agent ...:
        graph.add_node(f"{m.name}.after_agent", ...)
```

---

### Step 3：确定入口节点（factory.py:1406）

```python
# entry_node：整个 agent 生命周期只跑一次
if middleware_w_before_agent:
    entry_node = f"{middleware_w_before_agent[0].name}.before_agent"
elif middleware_w_before_model:
    entry_node = f"{middleware_w_before_model[0].name}.before_model"
else:
    entry_node = "model"

# loop_entry_node：tools 执行完后循环回这里
if middleware_w_before_model:
    loop_entry_node = f"{middleware_w_before_model[0].name}.before_model"
else:
    loop_entry_node = "model"
```

**关键区分：**
- `before_agent` → `entry_node`：**只在整个 agent 开始时跑一次**
- `before_model` → `loop_entry_node`：**每次调用 LLM 前都跑**（包括 tools 循环后的下一轮）

---

### Step 4：`wrap_model_call` 的特殊性（洋葱组合）

它不是图节点，而是在 `model` 节点**内部**组合成洋葱链（factory.py:956）：

```python
# 将多个 wrap_model_call 组合为单个洋葱链
sync_handlers = [m.wrap_model_call for m in middleware_w_wrap_model_call]
wrap_model_call_handler = _chain_model_call_handlers(sync_handlers)

# model 节点内部执行时：
result = wrap_model_call_handler(request, _execute_model)
# 等价于：MW1(request, lambda r: MW2(r, lambda r: _execute_model(r)))
```

这是真正的**洋葱模型**，可以在 LLM 调用前后各插入逻辑，Commands 从内层向外层累积。

---

### 最终图结构总览

```
invoke()
   ↓
[MW.before_agent]      ← Graph 节点，entry_node，运行一次
   ↓
[MW.before_model]      ← Graph 节点，loop_entry_node，每轮运行
   ↓
[model]                ← Graph 节点；wrap_model_call 在节点内部包裹（不是独立节点）
   ↓
[MW.after_model]       ← Graph 节点，每轮运行
   ↓
[tools]                ← Graph 节点；wrap_tool_call 在节点内部包裹
   ↓ (有 tool_call → 循环回 loop_entry_node；否则退出)
[MW.after_agent]       ← Graph 节点，exit_node，运行一次
```

## 5. invoke() 的调用链

`agent.invoke()` 本身不包含任何中间件逻辑，它只是 **驱动 Graph 执行的入口**。

源码位置：`langgraph/pregel/main.py:3047`（`CompiledStateGraph` 类）

### invoke 实现：stream 的包装器

```python
def invoke(self, input, config=None, *, stream_mode="values", ...):
    latest = None
    for chunk in self.stream(
        input, config,
        stream_mode=["updates", "values"],  # 内部同时订阅两种模式
        ...
    ):
        mode, payload = chunk
        if mode == "values":
            latest = payload   # 只保留最新的完整 state 快照
    return latest              # 返回最终 state
```

`invoke` = `stream` 的语法糖，收集所有 chunk，只返回最后一个完整 state。

### 完整调用链

```
agent.invoke({"messages": [...]}, config)
   │
   └── CompiledStateGraph.invoke()        # pregel/main.py:3047
          │  （invoke 只是包装，真正执行靠 stream）
          └── self.stream()               # Pregel 流式执行引擎
                 │
                 └── 按 Graph 边顺序调度各节点
                        │
                        ├── entry_node: [MW.before_agent]   ← 运行一次
                        │      （中间件逻辑在此执行，e.g. 消息压缩）
                        ├── loop:
                        │   ├── [MW.before_model]           ← 每轮
                        │   ├── [model]                     ← LLM 调用
                        │   │      └── wrap_model_call 洋葱链在内部执行
                        │   ├── [MW.after_model]            ← 每轮
                        │   └── [tools] ──────────────────┐
                        │          └── wrap_tool_call 洋葱链 │
                        │   ┌──────────────────────────────┘ loop back
                        │   └── （无 tool_call → 退出循环）
                        └── exit_node: [MW.after_agent]     ← 运行一次
```

**核心要点：** 中间件逻辑在 `create_agent` 时已编译进图节点，`invoke` 调用时 Pregel 引擎只需按拓扑顺序调度这些节点，不需要感知中间件的存在。